# Antardrishti — Sen1Floods11 evaluation

**Question this notebook answers:** how accurate is our Sentinel-1 flood detection, and is the `[-25, -12] dB` clamp in `backend/detection/sar.py` helping or hurting?

Context: on Kerala, August 2018, Otsu returned **-10.99 dB** and was clamped to **-12.0 dB**, giving a flood extent of 1,246.8 km². A synthetic test suggests -12 dB has precision around 0.6 — it catches all water and a lot of wet soil besides. This notebook measures that against hand-labelled ground truth.

Dataset: [Sen1Floods11](https://github.com/cloudtostreet/Sen1Floods11) — 446 hand-labelled 512×512 chips, 11 flood events, 6 continents, with Sentinel-1 VV/VH and matching water labels.

**Run this in Colab**, not locally — the data lives in a Google Cloud bucket and Colab is on the same network.

In [ ]:
!pip install -q rasterio

from google.colab import auth
auth.authenticate_user()
print("authenticated")

## 1. Look before assuming

Paths in this bucket have changed between dataset versions. List first, then set the paths in the next cell to whatever actually exists.

In [ ]:
!gsutil ls gs://sen1floods11/
print("---- one level down ----")
!gsutil ls gs://sen1floods11/v1.1/ 2>/dev/null || echo "(no v1.1)"
print("---- hand labelled ----")
!gsutil ls gs://sen1floods11/v1.1/data/flood_events/HandLabeled/ 2>/dev/null || echo "(path differs - read the listing above)"

In [ ]:
# Adjust these to match the listing above.
BUCKET = "gs://sen1floods11"
S1_PATH = f"{BUCKET}/v1.1/data/flood_events/HandLabeled/S1Hand"
LABEL_PATH = f"{BUCKET}/v1.1/data/flood_events/HandLabeled/LabelHand"

LOCAL = "/content/sen1floods11"
!mkdir -p {LOCAL}/s1 {LOCAL}/label

# Hand-labelled subset only: 446 chips, a few GB. The weakly-labelled set is
# far larger and is not needed to measure accuracy.
!gsutil -m cp -r {S1_PATH}/* {LOCAL}/s1/ 2>&1 | tail -3
!gsutil -m cp -r {LABEL_PATH}/* {LOCAL}/label/ 2>&1 | tail -3

!echo "S1 chips:" && ls {LOCAL}/s1 | wc -l
!echo "labels  :" && ls {LOCAL}/label | wc -l

## 2. Pull in our own thresholding code

These are numpy mirrors of `backend/detection/sar.py`, so the evaluation scores the algorithm that actually ships.

In [ ]:
# Option A - clone the repo:
# !git clone <your-repo-url> /content/antardrishti
# import sys; sys.path.insert(0, "/content/antardrishti/application/antardrishti/evaluation")

# Option B - upload evaluation/thresholds.py and evaluation/metrics.py:
from google.colab import files
print("Upload thresholds.py and metrics.py from application/antardrishti/evaluation/")
files.upload()

In [ ]:
import glob, os
import numpy as np
import rasterio

from thresholds import METHODS
from metrics import evaluate, aggregate, format_table

def chip_pairs(local=LOCAL):
    """Match S1 chips to their labels by the shared prefix."""
    labels = {}
    for path in glob.glob(f"{local}/label/*.tif"):
        key = os.path.basename(path).replace("_LabelHand.tif", "")
        labels[key] = path

    pairs = []
    for path in sorted(glob.glob(f"{local}/s1/*.tif")):
        key = os.path.basename(path).replace("_S1Hand.tif", "")
        if key in labels:
            pairs.append((key, path, labels[key]))
    return pairs

pairs = chip_pairs()
print(f"{len(pairs)} matched chip/label pairs")
print(pairs[0] if pairs else "NONE - check the filename suffixes above")

In [ ]:
def load(s1_path, label_path, band=1):
    """band 1 = VV, band 2 = VH. Values are dB.
    Labels: -1 nodata, 0 not water, 1 water."""
    with rasterio.open(s1_path) as src:
        image = src.read(band).astype(np.float64)
    with rasterio.open(label_path) as src:
        truth = src.read(1).astype(np.int8)
    return image, truth

image, truth = load(*pairs[0][1:])
print("image shape", image.shape, "| dB range", np.nanmin(image).round(1), "to", np.nanmax(image).round(1))
print("label values", np.unique(truth))
print("water fraction of labelled pixels:",
      round(float((truth == 1).sum() / max((truth != -1).sum(), 1)), 4))

## 3. Score every method over every chip

This is the table that replaces the six pass/fail rows currently in Chapter 7.

In [ ]:
from collections import defaultdict

per_method = defaultdict(list)
threshold_log = defaultdict(list)
clamp_counts = defaultdict(int)

for i, (key, s1_path, label_path) in enumerate(pairs):
    try:
        image, truth = load(s1_path, label_path)
    except Exception as exc:
        print(f"skip {key}: {exc}")
        continue

    if (truth != -1).sum() == 0:
        continue

    for name, method in METHODS.items():
        mask, info = method(image)
        per_method[name].append(evaluate(mask, truth))
        threshold_log[name].append(info["threshold"])
        if info.get("source") == "otsu_clamped":
            clamp_counts[name] += 1

    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(pairs)} chips")

results = {name: aggregate(chips) for name, chips in per_method.items()}
print()
print(format_table(results, "Sen1Floods11 hand-labelled test set"))

In [ ]:
print(f"{'method':<28}{'mean thr':>10}{'std':>8}{'clamped':>10}")
print("-" * 56)
for name in METHODS:
    values = np.array(threshold_log[name])
    clamped = clamp_counts.get(name, 0)
    share = f"{clamped / len(values) * 100:.0f}%" if len(values) else "-"
    print(f"{name:<28}{values.mean():>10.2f}{values.std():>8.2f}{share:>10}")

print("\nIf the clamped share is high, the bound is deciding rather than the data.")
print("On Kerala it clamped to exactly -12.0, which is what prompted this check.")

In [ ]:
# Micro vs macro. A wide gap means performance depends on how much water is
# present - itself worth reporting.
print(f"{'method':<28}{'micro IoU':>11}{'macro IoU':>11}{'gap':>8}")
print("-" * 58)
for name, agg in results.items():
    micro, macro = agg["micro"]["iou"], agg["macro"]["iou"]
    print(f"{name:<28}{micro:>11.3f}{macro:>11.3f}{micro - macro:>8.3f}")

## 4. Sweep the threshold

Which fixed threshold is actually best? This gives the empirical answer instead of the textbook one, and tells you where to set the clamp.

In [ ]:
from thresholds import apply_threshold

sweep = {}
for db in np.arange(-24, -9, 1.0):
    chips = []
    for key, s1_path, label_path in pairs:
        try:
            image, truth = load(s1_path, label_path)
        except Exception:
            continue
        if (truth != -1).sum() == 0:
            continue
        chips.append(evaluate(apply_threshold(image, db), truth))
    sweep[float(db)] = aggregate(chips)

print(f"{'threshold dB':>13}{'IoU':>8}{'F1':>8}{'prec':>8}{'recall':>8}")
print("-" * 45)
best = None
for db, agg in sweep.items():
    m = agg["micro"]
    print(f"{db:>13.0f}{m['iou']:>8.3f}{m['f1']:>8.3f}{m['precision']:>8.3f}{m['recall']:>8.3f}")
    if best is None or m["iou"] > best[1]:
        best = (db, m["iou"])

print(f"\nBest fixed threshold: {best[0]:.0f} dB at IoU {best[1]:.3f}")
print("Compare against Prithvi-EO-2.0 at 83.1% water IoU on this dataset.")

In [ ]:
import json

payload = {
    "dataset": "Sen1Floods11 HandLabeled",
    "chips_scored": len(pairs),
    "band": "VV",
    "methods": {
        name: {
            "micro": agg["micro"],
            "macro": agg["macro"],
            "mean_threshold_db": float(np.mean(threshold_log[name])),
            "clamped_share": clamp_counts.get(name, 0) / max(len(threshold_log[name]), 1),
        }
        for name, agg in results.items()
    },
    "threshold_sweep": {str(db): agg["micro"] for db, agg in sweep.items()},
}

with open("sen1floods11_results.json", "w") as fh:
    json.dump(payload, fh, indent=2)

from google.colab import files
files.download("sen1floods11_results.json")
print("Put this in models/ or results/ and cite it from the report.")

## What to do with the numbers

1. **If the best fixed threshold is well below -12 dB**, the upper clamp in `backend/detection/sar.py` is too permissive and the Kerala figure of 1,246.8 km² is an over-estimate. Move `WATER_DB_MAX` to the empirical value.
2. **If unbounded Otsu beats the clamped version**, drop the clamp and handle the pathological cases some other way.
3. **If the clamped share is high**, the bound is doing the deciding rather than the data — say so in the report rather than presenting Otsu as adaptive.
4. **Whatever the numbers**, this table is Chapter 7. Replace the six pass/fail test cases with it.

Reference point: Prithvi-EO-2.0-600M reports **83.1% water IoU** here. A simple threshold will score below that — but the Prithvi-CAFE paper found geospatial foundation models barely beat a plain U-Net on the held-out site (72.4 vs 70.6 IoU), so the gap may be smaller than expected. Reporting your baseline honestly against a published number is worth more than a big number with no baseline.